# FireSight-IR | Module 3a — Extensions

**Project:** FireSight-IR — FireSat Protoflight-aligned wildfire detection pipeline  
**Author:** Emmanuel Ibekwe | github.com/Ibekwemmanuel7  
**Depends on:** Module 3a (`03a_pinn_training_final.ipynb`) — best checkpoint must exist at `firesight_pinn_best.pt`

---

## What this notebook adds

| Section | Objective |
|---|---|
| A | **Probability score distributions** — wildfire vs false-alarm histogram |
| B | **Ablation study** — retrain 3 stripped variants and compare to full model |
| C | **Extended metrics** — wildfire precision + annotated confusion matrix |

> **Prerequisites:** Run Sections 0–6 of Module 3a first in this same session, OR re-run the shared setup cells below (Sections 0–6 are reproduced here for standalone use).


---
## Section 0 — Environment (same as Module 3a)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
!pip install torch tqdm scikit-learn matplotlib numpy pandas h5py pyarrow scipy -q

import os, json, time, warnings, copy
import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.cuda.amp import GradScaler, autocast
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_recall_fscore_support
)
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HAS_AMP = device.type == 'cuda'
print(f'Device : {device}')
if HAS_AMP:
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'PyTorch: {torch.__version__}')

---
## Section 1 — Configuration (same as Module 3a)

In [ ]:
BASE_DIR     = '/content/drive/MyDrive/firesight-ir'
ARCHIVE_PATH = f'{BASE_DIR}/data/processed/patches/firesight_patches.h5'
SPLIT_DIR    = f'{BASE_DIR}/data/splits'
MODEL_DIR    = f'{BASE_DIR}/models'
FIGURES_DIR  = f'{BASE_DIR}/figures'
CACHE_DIR    = f'{BASE_DIR}/data/cache'

WEIGHTS_PATH = f'{BASE_DIR}/data/scalers/class_weights_v2.json'
if not os.path.exists(WEIGHTS_PATH):
    WEIGHTS_PATH = f'{BASE_DIR}/data/scalers/class_weights.json'

for d in [MODEL_DIR, FIGURES_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

N_ATM=16; N_SRF=20; N_DERIVED=6; PATCH_SIZE=32; N_CHANNELS=4; N_CLASSES=3
BATCH_SIZE=1024; N_EPOCHS=30; LR_MAX=3e-4; WEIGHT_DECAY=1e-4; DROPOUT=0.3
NUM_WORKERS=2
LAMBDA_BL=0.10; LAMBDA_DR=0.05; LAMBDA_TH=0.05
BT_I4_FIRE_MIN=310.0; BTD_FIRE_MIN=10.0

BEST_PATH  = f'{MODEL_DIR}/firesight_pinn_best.pt'
LOG_PATH   = f'{MODEL_DIR}/training_log.json'

CACHE_FILES = {
    'patches': f'{CACHE_DIR}/patches.npy',
    'atm'    : f'{CACHE_DIR}/atm.npy',
    'srf'    : f'{CACHE_DIR}/srf.npy',
    'derived': f'{CACHE_DIR}/derived.npy',
    'labels' : f'{CACHE_DIR}/labels.npy',
    'aux'    : f'{CACHE_DIR}/aux.npy',
}

SEED=42
torch.manual_seed(SEED); np.random.seed(SEED)
if HAS_AMP: torch.cuda.manual_seed(SEED)
print('Configuration set.')

# ── Plot theme (matches Module 3a) ────────────────────────────────────────────
BG,PANEL='#0D1117','#161B22'
TEXT,SUBTEXT,BORDER='#E6EDF3','#8B949E','#30363D'
plt.rcParams.update({
    'figure.facecolor':BG,'axes.facecolor':PANEL,
    'axes.edgecolor':BORDER,'text.color':TEXT,
    'xtick.color':SUBTEXT,'ytick.color':SUBTEXT,
    'axes.labelcolor':SUBTEXT,'grid.color':BORDER
})

---
## Sections 2–6 — Dataset, Model, Loss, Loaders (reproduced for standalone run)

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
class FireSightDataset(Dataset):
    def __init__(self, cache_files, indices, augment=False):
        self.indices=np.sort(indices); self.augment=augment
        self.patches=np.load(cache_files['patches'],mmap_mode='r')
        self.atm    =np.load(cache_files['atm'],    mmap_mode='r')
        self.srf    =np.load(cache_files['srf'],    mmap_mode='r')
        self.derived=np.load(cache_files['derived'],mmap_mode='r')
        self.labels =np.load(cache_files['labels'], mmap_mode='r')
        self.aux    =np.load(cache_files['aux'],    mmap_mode='r')

    def __len__(self): return len(self.indices)

    def __getitem__(self, i):
        idx=int(self.indices[i]); patch=self.patches[idx].copy()
        if self.augment:
            k=np.random.randint(0,4)
            if k: patch=np.rot90(patch,k,axes=(1,2)).copy()
            if np.random.rand()>0.5: patch=np.flip(patch,axis=2).copy()
        return (
            torch.from_numpy(patch),
            torch.from_numpy(self.atm[idx].copy()),
            torch.from_numpy(self.srf[idx].copy()),
            torch.from_numpy(self.derived[idx].copy()),
            torch.tensor(int(self.labels[idx]),dtype=torch.long),
            torch.from_numpy(self.aux[idx].copy()),
        )

# ── Model ─────────────────────────────────────────────────────────────────────
class ResidualBlock(nn.Module):
    def __init__(self,in_dim,out_dim,dropout=0.2):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(in_dim,out_dim),nn.BatchNorm1d(out_dim),nn.ReLU(),
            nn.Dropout(dropout),nn.Linear(out_dim,out_dim),nn.BatchNorm1d(out_dim))
        self.proj=nn.Linear(in_dim,out_dim) if in_dim!=out_dim else nn.Identity()
    def forward(self,x): return F.relu(self.net(x)+self.proj(x))

class CNNBranch(nn.Module):
    def __init__(self,in_ch=4,dropout=0.2):
        super().__init__()
        self.enc=nn.Sequential(
            nn.Conv2d(in_ch,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(True),
            nn.Conv2d(32,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(True),
            nn.MaxPool2d(2),nn.Dropout2d(0.1),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(True),
            nn.Conv2d(64,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(True),
            nn.MaxPool2d(2),nn.Dropout2d(0.1),
            nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(True),
            nn.MaxPool2d(2))
        self.gap=nn.AdaptiveAvgPool2d(1); self.drop=nn.Dropout(dropout)
    def forward(self,x): return self.drop(self.gap(self.enc(x)).flatten(1))

class FireSightPINN(nn.Module):
    def __init__(self,n_atm=16,n_srf=20,n_der=6,n_cls=3,drop=0.3):
        super().__init__()
        self.cnn=CNNBranch(4,drop)
        self.atm=nn.Sequential(ResidualBlock(n_atm,64),ResidualBlock(64,32))
        self.srf=nn.Sequential(ResidualBlock(n_srf,64),ResidualBlock(64,32))
        self.der=nn.Sequential(
            nn.Linear(n_der,32),nn.BatchNorm1d(32),nn.ReLU(),nn.Dropout(0.1),
            nn.Linear(32,16),nn.BatchNorm1d(16),nn.ReLU())
        self.fusion=nn.Sequential(
            nn.Linear(208,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(drop),
            nn.Linear(128,64),nn.BatchNorm1d(64),nn.ReLU(),nn.Dropout(drop))
        self.cls  =nn.Linear(64,n_cls)
        self.trans=nn.Sequential(nn.Linear(64,16),nn.ReLU(),nn.Linear(16,1),nn.Sigmoid())

    def forward(self,patch,atm,srf,der):
        feat=self.fusion(torch.cat([self.cnn(patch),self.atm(atm),self.srf(srf),self.der(der)],dim=1))
        return self.cls(feat),self.trans(feat)

# ── Physics loss ──────────────────────────────────────────────────────────────
class PINNLoss(nn.Module):
    def __init__(self,weights,lbl=0.1,ldr=0.05,lth=0.05,bt_min=310.,btd_min=10.,device='cpu'):
        super().__init__()
        self.lbl,self.ldr,self.lth=lbl,ldr,lth
        self.bt_min,self.btd_min=bt_min,btd_min
        w=torch.tensor(weights,dtype=torch.float32).to(device)
        self.ce=nn.CrossEntropyLoss(weight=w); self.mse=nn.MSELoss()
    def forward(self,logits,trans,labels,aux):
        L_ce=self.ce(logits,labels)
        L_bl=self.mse(trans,aux[:,2:3])
        p_wf=F.softmax(logits,dim=1)[:,1]
        L_dr=(p_wf*(aux[:,0]<self.bt_min).float()).mean()
        L_th=(p_wf*(aux[:,1]<self.btd_min).float()).mean()
        total=L_ce+self.lbl*L_bl+self.ldr*L_dr+self.lth*L_th
        return total,{'ce':L_ce.item(),'bl':L_bl.item(),'dr':L_dr.item(),'th':L_th.item()}

print('Model, Dataset, PINNLoss defined.')

In [ ]:
# ── Splits and DataLoaders ────────────────────────────────────────────────────
train_idx = np.load(f'{SPLIT_DIR}/train_index.npy')
val_idx   = np.load(f'{SPLIT_DIR}/val_index.npy')
test_idx  = np.load(f'{SPLIT_DIR}/test_index.npy')
print(f'Splits — Train:{len(train_idx):,} Val:{len(val_idx):,} Test:{len(test_idx):,}')

with open(WEIGHTS_PATH) as f: cw=json.load(f)
if isinstance(cw,list): class_weights=cw
elif isinstance(cw,dict):
    class_weights=cw.get('_list',[float(cw.get(str(i),[6.9,0.36,9.81][i])) for i in range(3)])
else: class_weights=[6.9,0.36,9.81]
print(f'Class weights: {[round(w,3) for w in class_weights]}')

pin = device.type == 'cuda'
lkw = dict(num_workers=NUM_WORKERS, pin_memory=pin,
           prefetch_factor=2 if NUM_WORKERS>0 else None, persistent_workers=False)

val_ds   = FireSightDataset(CACHE_FILES, val_idx,   augment=False)
test_ds  = FireSightDataset(CACHE_FILES, test_idx,  augment=False)
train_ds = FireSightDataset(CACHE_FILES, train_idx, augment=True)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False, **lkw)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2, shuffle=False, **lkw)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,   shuffle=True,  **lkw)

print(f'Loaders ready.')

In [ ]:
# ── Load best checkpoint ──────────────────────────────────────────────────────
assert os.path.exists(BEST_PATH), f'Best checkpoint not found at {BEST_PATH}. Run Module 3a training first.'

full_model = FireSightPINN(N_ATM, N_SRF, N_DERIVED, N_CLASSES, DROPOUT).to(device)
ckpt = torch.load(BEST_PATH, map_location=device, weights_only=False)
full_model.load_state_dict(ckpt['model_state_dict'])
full_model.eval()
print(f'Full model loaded — epoch {ckpt["epoch"]} | val_loss={ckpt["val_loss"]:.4f}')

@torch.no_grad()
def collect_outputs(model, loader, max_n=None):
    """Returns (preds, labels, probs) arrays for the given loader."""
    model.eval()
    preds_l, labels_l, probs_l = [], [], []
    n = 0
    for patch, atm, srf, der, labels, aux in loader:
        if max_n and n >= max_n: break
        with autocast(enabled=HAS_AMP):
            logits, _ = model(
                patch.to(device), atm.to(device),
                srf.to(device),   der.to(device)
            )
        probs = F.softmax(logits, dim=1).cpu().numpy()
        preds_l.append(logits.argmax(1).cpu().numpy())
        labels_l.append(labels.numpy())
        probs_l.append(probs)
        n += len(labels)
    return (np.concatenate(preds_l), np.concatenate(labels_l),
            np.concatenate(probs_l))

print('Collecting val predictions...')
val_preds, val_labels, val_probs = collect_outputs(full_model, val_loader)
print(f'Val: {len(val_labels):,} samples collected.')

---
## Section A — Probability Score Distributions

Shows the model's **wildfire probability score** (P(wildfire)) separately for ground-truth wildfire pixels and ground-truth false-alarm pixels on the 2023 val set.  
The key result: if the two distributions are non-overlapping, AUC = 1.0000 is genuine — not a reporting artefact.

In [ ]:
# ── Wildfire probability scores by ground-truth class ─────────────────────────
p_wf_given_wf = val_probs[val_labels == 1, 1]   # P(wildfire) for true wildfire pixels
p_wf_given_fa = val_probs[val_labels == 2, 1]   # P(wildfire) for true false-alarm pixels
p_wf_given_nf = val_probs[val_labels == 0, 1]   # P(wildfire) for true non-fire pixels

print(f'True wildfire   → median P(wf)={np.median(p_wf_given_wf):.4f}  n={len(p_wf_given_wf):,}')
print(f'True false-alarm → median P(wf)={np.median(p_wf_given_fa):.4f}  n={len(p_wf_given_fa):,}')
print(f'True non-fire   → median P(wf)={np.median(p_wf_given_nf):.4f}  n={len(p_wf_given_nf):,}')

# Overlap check: fraction of FA pixels with P(wf) > 0.5
fa_leak = (p_wf_given_fa > 0.5).mean()
print(f'\nFalse-alarm leakage (P(wf)>0.5): {fa_leak*100:.2f}% of FA pixels')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6), facecolor=BG)
fig.patch.set_facecolor(BG)

BINS = np.linspace(0, 1, 51)

# ── Left: wildfire vs false-alarm (main result) ───────────────────────────────
ax = axes[0]
ax.set_facecolor(PANEL)
ax.hist(p_wf_given_wf, bins=BINS, alpha=0.75, color='#E85D04',
        density=True, label=f'True wildfire  (n={len(p_wf_given_wf):,})')
ax.hist(p_wf_given_fa, bins=BINS, alpha=0.75, color='#D97706',
        density=True, label=f'True false-alarm (n={len(p_wf_given_fa):,})')
ax.axvline(0.5, color='white', lw=1.5, ls='--', label='threshold = 0.5')
ax.set_xlabel('P(wildfire)', color=SUBTEXT)
ax.set_ylabel('Density', color=SUBTEXT)
ax.set_title('Wildfire vs False-Alarm\nP(wildfire) score distributions — Val 2023',
             color=TEXT, fontsize=11)
ax.legend(facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=9)
ax.grid(alpha=0.2)
ax.spines[['top', 'right']].set_visible(False)

# Annotate median markers
ax.axvline(np.median(p_wf_given_wf), color='#E85D04', lw=1, ls=':',
           label=f'WF median={np.median(p_wf_given_wf):.3f}')
ax.axvline(np.median(p_wf_given_fa), color='#D97706', lw=1, ls=':',
           label=f'FA median={np.median(p_wf_given_fa):.3f}')
ax.legend(facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)

# ── Right: all three classes stacked ─────────────────────────────────────────
ax = axes[1]
ax.set_facecolor(PANEL)
ax.hist(p_wf_given_wf, bins=BINS, alpha=0.65, color='#E85D04',
        density=True, label=f'Wildfire  (n={len(p_wf_given_wf):,})')
ax.hist(p_wf_given_fa, bins=BINS, alpha=0.65, color='#D97706',
        density=True, label=f'False-alarm (n={len(p_wf_given_fa):,})')
ax.hist(p_wf_given_nf, bins=BINS, alpha=0.65, color='#3A5C8A',
        density=True, label=f'Non-fire  (n={len(p_wf_given_nf):,})')
ax.axvline(0.5, color='white', lw=1.5, ls='--', label='threshold = 0.5')
ax.set_xlabel('P(wildfire)', color=SUBTEXT)
ax.set_ylabel('Density', color=SUBTEXT)
ax.set_title('All Classes\nP(wildfire) score distributions — Val 2023',
             color=TEXT, fontsize=11)
ax.legend(facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=9)
ax.grid(alpha=0.2)
ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('FireSight-IR | Module 3a Extension A — Probability Score Distributions',
             color=TEXT, fontsize=13, y=1.01)
plt.tight_layout()
out = f'{FIGURES_DIR}/03a_ext_score_distributions.png'
plt.savefig(out, dpi=160, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Saved → {out}')

---
## Section B — Ablation Study

Retrain 3 stripped variants from scratch and compare to the full model.  
Each variant **removes one input branch** by zeroing out its contribution to the fusion layer.

| Variant | What's removed | Fusion input dim |
|---|---|---|
| No physics loss | Physics terms in loss (BL + DR + TH → 0) | 208 (unchanged) |
| No ERA5 | ATM branch → zeros | 208 (atm zeros) |
| No surface | SRF branch → zeros | 208 (srf zeros) |
| Full model | — | 208 |

> **Runtime:** Each variant trains for `ABLATION_EPOCHS` epochs (~5–8 min each on T4).  
> Set `ABLATION_EPOCHS = 30` to match the full model exactly, or use 15 for a fast comparison.


In [ ]:
ABLATION_EPOCHS = 15   # increase to 30 for full comparison; 15 is good for fast ablation

# ── Ablation model wrappers ───────────────────────────────────────────────────
# Strategy: same FireSightPINN architecture, but we zero out specific branch
# inputs at the dataset level. This avoids modifying the architecture and
# means the fusion MLP sees the same shape, just with zeroed branches.

class AblationDataset(Dataset):
    """
    Wraps FireSightDataset and zeros out specified branches.
    zero_atm=True → ERA5 branch receives all-zeros
    zero_srf=True → Surface branch receives all-zeros
    """
    def __init__(self, base_ds, zero_atm=False, zero_srf=False):
        self.base = base_ds
        self.zero_atm = zero_atm
        self.zero_srf = zero_srf

    def __len__(self): return len(self.base)

    def __getitem__(self, i):
        patch, atm, srf, der, label, aux = self.base[i]
        if self.zero_atm: atm = torch.zeros_like(atm)
        if self.zero_srf: srf = torch.zeros_like(srf)
        return patch, atm, srf, der, label, aux


class CEOnlyLoss(nn.Module):
    """Cross-entropy only — no physics constraints. Used for no-physics ablation."""
    def __init__(self, weights, device='cpu'):
        super().__init__()
        w = torch.tensor(weights, dtype=torch.float32).to(device)
        self.ce = nn.CrossEntropyLoss(weight=w)

    def forward(self, logits, trans, labels, aux):
        loss = self.ce(logits, labels)
        return loss, {'ce': loss.item(), 'bl': 0., 'dr': 0., 'th': 0.}


def train_ablation(variant_name, zero_atm=False, zero_srf=False,
                   use_physics=True, n_epochs=ABLATION_EPOCHS):
    """Train one ablation variant and return val metrics dict."""
    print(f'\n{"="*55}')
    print(f'  Ablation: {variant_name}')
    print(f'  zero_atm={zero_atm}  zero_srf={zero_srf}  physics={use_physics}')
    print(f'  Epochs: {n_epochs}')
    print(f'{"="*55}')

    torch.manual_seed(SEED); np.random.seed(SEED)
    if HAS_AMP: torch.cuda.manual_seed(SEED)

    # Datasets
    a_train = AblationDataset(train_ds, zero_atm=zero_atm, zero_srf=zero_srf)
    a_val   = AblationDataset(val_ds,   zero_atm=zero_atm, zero_srf=zero_srf)

    a_train_loader = DataLoader(a_train, batch_size=BATCH_SIZE,   shuffle=True,  **lkw)
    a_val_loader   = DataLoader(a_val,   batch_size=BATCH_SIZE*2, shuffle=False, **lkw)

    # Model (fresh weights)
    model = FireSightPINN(N_ATM, N_SRF, N_DERIVED, N_CLASSES, DROPOUT).to(device)

    # Loss
    if use_physics:
        criterion = PINNLoss(class_weights, LAMBDA_BL, LAMBDA_DR, LAMBDA_TH,
                             BT_I4_FIRE_MIN, BTD_FIRE_MIN, device)
    else:
        criterion = CEOnlyLoss(class_weights, device)

    optimizer = AdamW(model.parameters(), lr=LR_MAX, weight_decay=WEIGHT_DECAY)
    scheduler = OneCycleLR(optimizer, max_lr=LR_MAX,
                           steps_per_epoch=len(a_train_loader),
                           epochs=n_epochs, pct_start=0.1, anneal_strategy='cos')
    scaler = GradScaler(enabled=HAS_AMP)

    best_val_loss = float('inf')
    best_state    = None

    for epoch in range(n_epochs):
        # Train
        model.train()
        for patch, atm, srf, der, labels, aux in a_train_loader:
            patch  = patch.to(device,  non_blocking=True)
            atm    = atm.to(device,    non_blocking=True)
            srf    = srf.to(device,    non_blocking=True)
            der    = der.to(device,    non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            aux    = aux.to(device,    non_blocking=True)
            optimizer.zero_grad()
            with autocast(enabled=HAS_AMP):
                logits, trans = model(patch, atm, srf, der)
                loss, _ = criterion(logits, trans, labels, aux)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()

        # Validate
        model.eval()
        val_total = 0; vp, vl = [], []
        with torch.no_grad():
            for patch, atm, srf, der, labels, aux in a_val_loader:
                with autocast(enabled=HAS_AMP):
                    logits, trans = model(
                        patch.to(device), atm.to(device),
                        srf.to(device),   der.to(device)
                    )
                    v_loss, _ = criterion(logits, trans, labels.to(device), aux.to(device))
                val_total += v_loss.item() * len(labels)
                vp.append(logits.argmax(1).cpu().numpy())
                vl.append(labels.numpy())

        val_loss = val_total / len(val_idx)
        vp_arr = np.concatenate(vp); vl_arr = np.concatenate(vl)
        val_acc = (vp_arr == vl_arr).mean()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

        print(f'  Ep {epoch+1:02d}/{n_epochs} val_loss={val_loss:.4f} val_acc={val_acc:.4f}')

    # Final eval on best state
    model.load_state_dict(best_state)
    model.eval()
    all_preds, all_labels, all_probs = collect_outputs(model, a_val_loader)

    prec, rec, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average=None, labels=[0, 1, 2], zero_division=0
    )
    fa_auc = roc_auc_score((all_labels == 2).astype(int), all_probs[:, 2]) \
        if (all_labels == 2).sum() > 0 else 0.0
    wf_auc = roc_auc_score((all_labels == 1).astype(int), all_probs[:, 1])

    metrics = {
        'variant'      : variant_name,
        'val_acc'      : float((all_preds == all_labels).mean()),
        'wf_recall'    : float(rec[1]),
        'wf_precision' : float(prec[1]),
        'wf_f1'        : float(f1[1]),
        'fa_recall'    : float(rec[2]),
        'fa_precision' : float(prec[2]),
        'fa_auc'       : float(fa_auc),
        'wf_auc'       : float(wf_auc),
        'best_val_loss': float(best_val_loss),
    }
    print(f'\n  ✓ {variant_name} complete')
    print(f'    WF recall={metrics["wf_recall"]:.4f}  FA recall={metrics["fa_recall"]:.4f}  FA AUC={metrics["fa_auc"]:.4f}')
    return metrics

print('Ablation framework ready.')

In [ ]:
# ── Full model metrics (from loaded checkpoint — no retraining needed) ─────────
prec_full, rec_full, f1_full, _ = precision_recall_fscore_support(
    val_labels, val_preds, average=None, labels=[0, 1, 2], zero_division=0
)
fa_auc_full = roc_auc_score((val_labels == 2).astype(int), val_probs[:, 2])
wf_auc_full = roc_auc_score((val_labels == 1).astype(int), val_probs[:, 1])

full_metrics = {
    'variant'      : 'Full model',
    'val_acc'      : float((val_preds == val_labels).mean()),
    'wf_recall'    : float(rec_full[1]),
    'wf_precision' : float(prec_full[1]),
    'wf_f1'        : float(f1_full[1]),
    'fa_recall'    : float(rec_full[2]),
    'fa_precision' : float(prec_full[2]),
    'fa_auc'       : float(fa_auc_full),
    'wf_auc'       : float(wf_auc_full),
    'best_val_loss': float(ckpt['val_loss']),
}
print('Full model metrics:')
for k, v in full_metrics.items():
    print(f'  {k:<18}: {v}')

In [ ]:
# ── Run the 3 ablation variants ───────────────────────────────────────────────
# Each takes ~5-8 min on T4 at ABLATION_EPOCHS=15
# Total: ~20-25 min for all three

ablation_results = [full_metrics]  # start with full model as baseline

ablation_results.append(
    train_ablation('No physics loss', zero_atm=False, zero_srf=False, use_physics=False)
)
ablation_results.append(
    train_ablation('No ERA5 (ATM)', zero_atm=True, zero_srf=False, use_physics=True)
)
ablation_results.append(
    train_ablation('No surface (SRF)', zero_atm=False, zero_srf=True, use_physics=True)
)

# Save results
ablation_path = f'{MODEL_DIR}/ablation_results.json'
with open(ablation_path, 'w') as f:
    json.dump(ablation_results, f, indent=2)
print(f'\nAblation results saved → {ablation_path}')

In [ ]:
# ── Ablation comparison chart ─────────────────────────────────────────────────
variants   = [r['variant']       for r in ablation_results]
wf_recalls = [r['wf_recall']     for r in ablation_results]
fa_recalls = [r['fa_recall']     for r in ablation_results]
fa_aucs    = [r['fa_auc']        for r in ablation_results]
wf_precs   = [r['wf_precision']  for r in ablation_results]
overall    = [r['val_acc']       for r in ablation_results]

x = np.arange(len(variants))
bar_w = 0.16

fig, ax = plt.subplots(figsize=(14, 6), facecolor=BG)
ax.set_facecolor(PANEL)

bars = [
    (wf_recalls, '#E85D04', 'WF Recall'),
    (wf_precs,   '#FF8C42', 'WF Precision'),
    (fa_recalls, '#D97706', 'FA Recall'),
    (fa_aucs,    '#4CAF50', 'FA AUC'),
    (overall,    '#4FC3F7', 'Overall Acc'),
]

for i, (vals, color, label) in enumerate(bars):
    offset = (i - 2) * bar_w
    rects = ax.bar(x + offset, vals, bar_w, color=color, alpha=0.85, label=label)
    for rect, v in zip(rects, vals):
        ax.text(rect.get_x() + rect.get_width()/2, rect.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', color=TEXT, fontsize=7, rotation=45)

# Highlight full model column
ax.axvspan(-0.5, 0.5, alpha=0.06, color='white')
ax.text(0, 1.08, '★ baseline', ha='center', va='bottom',
        color=TEXT, fontsize=8, transform=ax.get_xaxis_transform())

ax.set_xticks(x)
ax.set_xticklabels(variants, color=TEXT, fontsize=10)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score', color=SUBTEXT)
ax.set_title('FireSight-IR | Ablation Study — Component Impact (Val 2023)',
             color=TEXT, fontsize=12)
ax.legend(facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT,
          fontsize=9, loc='upper right', ncol=3)
ax.grid(axis='y', alpha=0.2)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
out = f'{FIGURES_DIR}/03a_ext_ablation.png'
plt.savefig(out, dpi=160, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Saved → {out}')

# ── Print delta table ─────────────────────────────────────────────────────────
print('\nDelta from full model (negative = degradation):')
print(f'{"Variant":<22} {"WF Rec":>8} {"WF Prec":>8} {"FA Rec":>8} {"FA AUC":>8} {"Acc":>8}')
print('-' * 66)
base = ablation_results[0]
for r in ablation_results:
    delta_wfr  = r['wf_recall']    - base['wf_recall']
    delta_wfp  = r['wf_precision'] - base['wf_precision']
    delta_far  = r['fa_recall']    - base['fa_recall']
    delta_faa  = r['fa_auc']       - base['fa_auc']
    delta_acc  = r['val_acc']      - base['val_acc']
    sign = lambda v: f'+{v:.4f}' if v >= 0 else f'{v:.4f}'
    print(f'{r["variant"]:<22} {sign(delta_wfr):>8} {sign(delta_wfp):>8} '
          f'{sign(delta_far):>8} {sign(delta_faa):>8} {sign(delta_acc):>8}')

---
## Section C — Extended Metrics

Adds **wildfire precision** to the existing evaluation and produces an annotated confusion matrix with absolute counts, row-normalised rates, and per-class summary statistics.

In [ ]:
# ── Full classification report with precision highlighted ─────────────────────
CNAMES = ['non-fire', 'wildfire', 'false-alarm']

def extended_metrics(preds, labels, probs, split_name):
    print(f'\n{"═"*60}')
    print(f'  {split_name}')
    print(f'{"═"*60}')
    print(classification_report(labels, preds, target_names=CNAMES, digits=4))

    prec, rec, f1, sup = precision_recall_fscore_support(
        labels, preds, average=None, labels=[0,1,2], zero_division=0
    )

    print('Key wildfire metrics:')
    print(f'  Wildfire precision : {prec[1]:.4f}  ({int(prec[1]*sup[1]):,} of {int(sup[1]):,} '
          f'predicted-wildfire are truly wildfire)')
    print(f'  Wildfire recall    : {rec[1]:.4f}  ({int(rec[1]*sup[1]):,} of {int(sup[1]):,} '
          f'true wildfires detected)')
    print(f'  Wildfire F1        : {f1[1]:.4f}')

    if (labels == 2).sum() > 0:
        fa_auc = roc_auc_score((labels==2).astype(int), probs[:,2])
        print(f'  False-alarm AUC    : {fa_auc:.4f}')
    return prec, rec, f1, sup

print('Running extended metrics on Val 2023...')
prec_v, rec_v, f1_v, sup_v = extended_metrics(val_preds,  val_labels,  val_probs,  'Val  (2023 holdout)')

In [ ]:
# ── Annotated confusion matrices: counts + rates side by side ─────────────────
def plot_confusion_matrix(preds, labels, title, ax_counts, ax_rates):
    cm_abs  = confusion_matrix(labels, preds, labels=[0,1,2])
    cm_norm = confusion_matrix(labels, preds, labels=[0,1,2], normalize='true')

    for ax, cm, fmt, vmax, subtitle in [
        (ax_counts, cm_abs,  'd',     None, 'Absolute counts'),
        (ax_rates,  cm_norm, '.3f',  1.0,  'Row-normalised (recall-view)'),
    ]:
        ax.set_facecolor(PANEL)
        im = ax.imshow(cm, cmap='Blues',
                       vmin=0, vmax=vmax if vmax else cm.max())
        ax.set_xticks(range(3)); ax.set_yticks(range(3))
        ax.set_xticklabels(CNAMES, rotation=30, ha='right', color=TEXT, fontsize=8)
        ax.set_yticklabels(CNAMES, color=TEXT, fontsize=8)
        ax.set_xlabel('Predicted', color=SUBTEXT)
        ax.set_ylabel('True', color=SUBTEXT)
        ax.set_title(f'{title}\n{subtitle}', color=TEXT, fontsize=10)

        for i in range(3):
            for j in range(3):
                val = cm[i, j]
                txt = f'{val:{fmt}}'
                # Add percentage annotation below count
                if fmt == 'd':
                    row_sum = cm[i].sum()
                    pct = val / row_sum * 100 if row_sum > 0 else 0
                    txt = f'{val:,}\n({pct:.1f}%)'
                bg_val = val / (vmax or cm.max())
                ax.text(j, i, txt,
                        ha='center', va='center', fontsize=8,
                        color='white' if bg_val < 0.5 else '#0D1117')
        plt.colorbar(im, ax=ax, fraction=0.046)


fig, axes = plt.subplots(2, 2, figsize=(14, 11), facecolor=BG)
fig.patch.set_facecolor(BG)

# Collect test predictions too for side-by-side comparison
print('Collecting test predictions...')
test_preds, test_labels, test_probs = collect_outputs(full_model, test_loader)

plot_confusion_matrix(test_preds, test_labels, 'Test (2018-2022)',
                      axes[0][0], axes[0][1])
plot_confusion_matrix(val_preds,  val_labels,  'Val (2023 holdout)',
                      axes[1][0], axes[1][1])

fig.suptitle('FireSight-IR | Module 3a Extension C — Confusion Matrices',
             color=TEXT, fontsize=13, y=1.01)
plt.tight_layout()
out = f'{FIGURES_DIR}/03a_ext_confusion_matrix.png'
plt.savefig(out, dpi=160, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Saved → {out}')

In [ ]:
# ── Per-class metric bar chart (precision / recall / F1 side by side) ─────────
prec_t, rec_t, f1_t, sup_t = extended_metrics(
    test_preds, test_labels, test_probs, 'Test (2018-2022 holdout)'
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG)
fig.patch.set_facecolor(BG)

cls_colors = ['#3A5C8A', '#E85D04', '#D97706']
metric_labels = ['Precision', 'Recall', 'F1']
x_pos = np.arange(len(metric_labels))
bar_w = 0.25

for ax, (prec, rec, f1, split_name) in zip(axes, [
    (prec_t, rec_t, f1_t, 'Test (2018–2022)'),
    (prec_v, rec_v, f1_v, 'Val (2023)'),
]):
    ax.set_facecolor(PANEL)
    for ci, (cname, col) in enumerate(zip(CNAMES, cls_colors)):
        vals = [prec[ci], rec[ci], f1[ci]]
        offset = (ci - 1) * bar_w
        rects = ax.bar(x_pos + offset, vals, bar_w, color=col, alpha=0.85, label=cname)
        for rect, v in zip(rects, vals):
            ax.text(rect.get_x() + rect.get_width()/2, rect.get_height() + 0.008,
                    f'{v:.3f}', ha='center', va='bottom', color=TEXT, fontsize=8)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(metric_labels, color=TEXT)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score', color=SUBTEXT)
    ax.set_title(split_name, color=TEXT, fontsize=11)
    ax.legend(facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=9)
    ax.grid(axis='y', alpha=0.2)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('FireSight-IR | Module 3a Extension C — Per-Class Precision / Recall / F1',
             color=TEXT, fontsize=12, y=1.01)
plt.tight_layout()
out = f'{FIGURES_DIR}/03a_ext_per_class_metrics.png'
plt.savefig(out, dpi=160, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Saved → {out}')

---
## Summary

In [ ]:
print('=' * 60)
print('  FireSight-IR | Module 3a Extensions — Complete')
print('=' * 60)
outputs = [
    (f'{FIGURES_DIR}/03a_ext_score_distributions.png', 'A: Probability score distributions'),
    (f'{MODEL_DIR}/ablation_results.json',              'B: Ablation results (JSON)'),
    (f'{FIGURES_DIR}/03a_ext_ablation.png',             'B: Ablation comparison chart'),
    (f'{FIGURES_DIR}/03a_ext_confusion_matrix.png',     'C: Annotated confusion matrices'),
    (f'{FIGURES_DIR}/03a_ext_per_class_metrics.png',    'C: Per-class precision/recall/F1'),
]
for path, desc in outputs:
    e = '✓' if os.path.exists(path) else '✗ (not yet generated)'
    print(f'  {e}  {desc}')
print('=' * 60)